# 小量補抓與官方價格比對（Eligibility v0.1 後續）

依序跑第 1–8 格。第 9 格（富邦）要先把第 1 格的 `RUN_FUBON` 改成 `True` 才會執行。

| 格 | 內容 | 網路請求 |
|---|---|---|
| 5 | B 類 13 個交易所日期、97 個「5 分K 多於官方」的股票日離線診斷：多出的量是否等於某個成分、相鄰日是否有反向殘差、13:30 後是否有 K 棒、FinLab 是否一致 | 0 |
| 6 | 官方重抓 `refetch_minimal.csv` 列出的報表（BFT41U × 3 天），存到新的證據資料夾並與既有 shard 比對。既有 shard 不修改 | 3 |
| 7 | 官方日K 開高低收 vs 5 分K 彙總。官方價格取自已存的原始回應 | 0 |
| 8 | 帶入價格比對結果，重跑 Eligibility（只在 `PRICE_MODE="full"` 時執行） | 0 |
| 9 | 富邦日K / 5 分K / 1 分K 重抓 B 類股票日，只讀行情端點 | 約 291 |
| 10 | 複製結果到 Drive，路徑為 `audits/targeted_checks/` | 0 |

`PRICE_MODE`：
- `"smoke"`：3 檔、1 季，約 2–5 分鐘。
- `"full"`：全部標的、全期，需讀取全部 5 分K partitions，約 30–60 分鐘。可中斷，中斷後設定 `PRICE_RESUME_DIR` 即可續跑。

In [ ]:
# 1. 參數
REPO_OWNER, REPO_NAME, BRANCH = "hh4832", "tw-market-data-lake", "main"
DATA_ROOT = "/content/drive/MyDrive/00Quant_Research/data"
AUD = f"{DATA_ROOT}/_tw-market-data-lake/audits"
LEDGER_DIR = f"{AUD}/gap_ledger_20261026_163358_3e8eed0"
RECON_DIR = f"{AUD}/official_volume_batch_recon/volume_recon_20261027_181218"
ELIG_DIR = f"{AUD}/eligibility_v0.1/eligibility_20261029_212835_c0864ba"
DATASET_ROOT = f"{DATA_ROOT}/_tw-market-data-lake/reference/official_volume/v1"
PRICE_MODE = "smoke"            # "smoke" | "full"
PRICE_RESUME_DIR = ""           # full 模式中斷後，填入本機上次的 price_check_* 資料夾
RUN_FUBON = False               # True 才會執行第 9 格（約 291 個富邦行情請求）
OUT_DIR = "/content/tw_lake_targeted"
PROJECT_DIR = f"/content/{REPO_NAME}"

In [ ]:
# 2. 掛載 Google Drive
from google.colab import drive
drive.mount("/content/drive")

In [ ]:
# 3. clone / pull + 依賴 + 依賴
import subprocess, os
if os.path.exists(f"{PROJECT_DIR}/.git"):
    subprocess.run(["git", "-C", PROJECT_DIR, "fetch", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", PROJECT_DIR, "checkout", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--branch", BRANCH, f"https://github.com/{REPO_OWNER}/{REPO_NAME}.git", PROJECT_DIR], check=True)
print("HEAD:", subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "HEAD"], text=True).strip())
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml>=6", "tqdm>=4.66"], check=True)
def stream(args):
    env = dict(os.environ, PYTHONPATH=f"{PROJECT_DIR}/src", PYTHONUNBUFFERED="1")
    p = subprocess.Popen(args, cwd=PROJECT_DIR, env=env, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in p.stdout:
        print(line, end="")
    return p.wait()

def ensure_drive():
    """Mount Drive safely. If an earlier cell unmounted Drive and then wrote under /content/drive,
    those files are only on this runtime's disk: move them aside (never delete) and copy any audit
    folders that are missing on the real Drive."""
    import os, shutil, time
    from google.colab import drive
    if os.path.ismount("/content/drive"):
        return
    stray = None
    if os.path.isdir("/content/drive") and os.listdir("/content/drive"):
        stray = f"/content/drive_local_{int(time.time())}"
        shutil.move("/content/drive", stray)
        print("moved local (unsynced) files to", stray)
    drive.mount("/content/drive")
    if stray:
        src_root = f"{stray}/MyDrive/00Quant_Research/data/_tw-market-data-lake/audits"
        dst_root = f"{DATA_ROOT}/_tw-market-data-lake/audits"
        for dirpath, dirnames, filenames in os.walk(src_root):
            rel = os.path.relpath(dirpath, src_root)
            for d in list(dirnames):
                target = os.path.join(dst_root, rel, d)
                if not os.path.exists(target):
                    shutil.copytree(os.path.join(dirpath, d), target)
                    print("copied missing", target)
                    dirnames.remove(d)
print("Python", sys.version.split()[0], "(目標 3.11；不同版本時請留意套件相容性)")

In [ ]:
# 4. Preflight（路徑與輸入）
for p in (LEDGER_DIR, RECON_DIR, ELIG_DIR, DATASET_ROOT, f"{ELIG_DIR}/refetch_minimal.csv"):
    print("OK " if os.path.exists(p) else "MISSING ", p)
assert all(os.path.exists(p) for p in (LEDGER_DIR, RECON_DIR, ELIG_DIR, DATASET_ROOT))

In [ ]:
# 5. B 類離線診斷（無網路）
import glob
print("exit", stream([sys.executable, "-m", "tw_lake.audit.targeted_checks", "overcount", "--data-root", DATA_ROOT,
                      "--recon-dir", RECON_DIR, "--eligibility-dir", ELIG_DIR, "--out-dir", OUT_DIR]))
OVER_DIR = sorted(glob.glob(f"{OUT_DIR}/overcount_diag_*"))[-1]

In [ ]:
# 6. 官方重抓 refetch_minimal.csv（3 個請求，存新資料夾，不改既有 shard）
print("exit", stream([sys.executable, "-m", "tw_lake.audit.targeted_checks", "official", "--eligibility-dir", ELIG_DIR,
                      "--dataset-root", DATASET_ROOT, "--out-dir", OUT_DIR]))
REFETCH_DIR = sorted(glob.glob(f"{OUT_DIR}/official_refetch_*"))[-1]

In [ ]:
# 7. 官方價格比對（無網路）
args = [sys.executable, "-m", "tw_lake.audit.price_check", "--data-root", DATA_ROOT, "--ledger-dir", LEDGER_DIR,
        "--dataset-root", DATASET_ROOT, "--out-dir", OUT_DIR, "--workers", "12"]
if PRICE_MODE == "smoke":
    args += ["--symbols", "2330,2317,8299", "--start", "2024-07-01", "--end", "2024-09-30"]
if PRICE_RESUME_DIR:
    args += ["--resume-dir", PRICE_RESUME_DIR]
print("exit", stream(args))
PRICE_DIR = PRICE_RESUME_DIR or sorted(glob.glob(f"{OUT_DIR}/price_check_*"))[-1]

In [ ]:
# 8. 帶入價格比對重跑 Eligibility（只在 full 模式）
if PRICE_MODE == "full":
    print("exit", stream([sys.executable, "-m", "tw_lake.audit.eligibility", "--data-root", DATA_ROOT,
                          "--ledger-dir", LEDGER_DIR, "--recon-dir", RECON_DIR, "--dataset-root", DATASET_ROOT,
                          "--out-dir", OUT_DIR, "--price-check-dir", PRICE_DIR]))
else:
    print("smoke 模式：略過")

In [ ]:
# 9a. 富邦憑證（Colab Secrets，值不會被印出）＋ SDK；RUN_FUBON=False 時略過
if RUN_FUBON:
    from google.colab import userdata
    FUBON_PFX_DRIVE_PATH = "/content/drive/MyDrive/Colab Notebooks/Fubon_Cert/your_cert.pfx"   # 改成實際位置
    for k, req in (("FUBON_USER_ID", True), ("FUBON_PASSWORD", False), ("FUBON_CERT_PASSWORD", False), ("FUBON_API_KEY", False)):
        try:
            v = userdata.get(k)
        except Exception:
            v = None
        if req and not v:
            raise RuntimeError(f"缺少 Colab Secret: {k}")
        os.environ[k] = v or ""
    assert os.path.exists(FUBON_PFX_DRIVE_PATH), "找不到憑證檔，請修改 FUBON_PFX_DRIVE_PATH"
    os.environ["FUBON_CERT_PATH"] = FUBON_PFX_DRIVE_PATH
    subprocess.run([sys.executable, f"{PROJECT_DIR}/scripts/install_colab_dependencies.py"], check=True)
else:
    print('RUN_FUBON=False：略過富邦')


In [ ]:
# 9b. 富邦行情重抓 B 類股票日（只讀行情端點）
if RUN_FUBON:
    print("exit", stream([sys.executable, "-m", "tw_lake.audit.targeted_checks", "fubon", "--data-root", DATA_ROOT,
                          "--overcount-dir", OVER_DIR, "--out-dir", OUT_DIR, "--max-symbol-days", "120"]))

In [ ]:
# 10. 複製到 Drive（新資料夾，不覆寫）
import shutil
from datetime import datetime, timezone, timedelta
head = subprocess.check_output(["git", "-C", PROJECT_DIR, "rev-parse", "--short", "HEAD"], text=True).strip()
dest = f"{AUD}/targeted_checks/{datetime.now(timezone(timedelta(hours=8))):%Y%m%d_%H%M%S}_{head}"
for d in sorted(glob.glob(f"{OUT_DIR}/*")):
    if os.path.isdir(d):
        shutil.copytree(d, f"{dest}/{os.path.basename(d)}", ignore=shutil.ignore_patterns("chunks"))
print("copied to", dest)
from google.colab import drive
drive.flush_and_unmount()